In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob

# --- 1. Configuration ---

# Define the single path to your baseline model's results
BASELINE_CSV_PATH = '/kaggle/input/11-r-b-obb-w-yolov8n/yolo_training_results/yolov8n-obb/results.csv'

# Define the output directory for the plot
OUTPUT_PLOT_DIRECTORY = "/kaggle/working/comparison_plots"
os.makedirs(OUTPUT_PLOT_DIRECTORY, exist_ok=True)

# This section automatically finds and names models
# It uses keywords from the folder names to assign labels in the plot
KEYWORD_TO_NAME_MAPPING = {
    'off': 'RandWin (Offline)',
    'randwin': 'RandWin (Online)'
}

print("✅ Configuration loaded.")

# --- 2. Automatic Model Discovery ---
print("\n🔍 Starting automatic model discovery...")
MODELS_TO_COMPARE = {'Baseline': BASELINE_CSV_PATH}

try:
    # Find the top-level directory name of the baseline to exclude it from the search
    baseline_dir_name = BASELINE_CSV_PATH.split('/')[3]
    print(f"  - Baseline directory to exclude: '{baseline_dir_name}'")

    # Search for all possible results.csv files in the /kaggle/input directory
    search_pattern = '/kaggle/input/*/yolo_training_results/yolov8n-obb/results.csv'
    all_results_files = glob.glob(search_pattern)

    for file_path in all_results_files:
        if file_path == BASELINE_CSV_PATH:
            continue # Skip the baseline file itself

        # Extract the directory name for the current file
        current_dir_name = file_path.split('/')[3]

        # Find a matching keyword to name the model
        model_name = None
        for keyword, name in KEYWORD_TO_NAME_MAPPING.items():
            if keyword in current_dir_name:
                model_name = name
                break
        
        if model_name:
            MODELS_TO_COMPARE[model_name] = file_path
            print(f"  - Found and added '{model_name}'")
        else:
            print(f"  - ⚠️ WARNING: Found a results file in '{current_dir_name}' but couldn't map it to a name.")

except Exception as e:
    print(f"  - ❌ ERROR during model discovery: {e}")

print(f"\n✅ Discovery complete. Models to compare: {list(MODELS_TO_COMPARE.keys())}")


# Define the metrics to plot from the CSV files
METRICS_TO_PLOT = {
    'mAP@0.5': 'metrics/mAP50(B)',
    'mAP@0.5:0.95': 'metrics/mAP50-95(B)',
    'Precision': 'metrics/precision(B)',
    'Recall': 'metrics/recall(B)'
}

# --- 3. Data Loading ---

loaded_data = {}
for name, path in MODELS_TO_COMPARE.items():
    try:
        if os.path.exists(path):
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()
            loaded_data[name] = df
            print(f"  - Successfully loaded data for '{name}'")
        else:
            print(f"  - ⚠️ WARNING: File not found for '{name}' at path: {path}")
    except Exception as e:
        print(f"  - ❌ ERROR: Could not load or process file for '{name}'. Reason: {e}")

if not loaded_data or len(loaded_data) < 2:
    print("\n❌ FATAL: Fewer than two models were loaded. Cannot generate comparison plots.")
else:
    # --- 4. Plotting Logic ---
    print("\n🖼️  Generating comparison plot...")
    try:
        num_metrics = len(METRICS_TO_PLOT)
        fig, axes = plt.subplots(num_metrics, 1, figsize=(14, 7 * num_metrics), sharex=True)
        fig.suptitle('Baseline vs. Random Window Augmentation', fontsize=20, weight='bold')

        # --- THIS SECTION HAS BEEN UPDATED ---
        # Define a specific color for each augmentation type
        aug_color_map = {
            'RandWin (Online)': 'orange',
            'RandWin (Offline)': 'blue'
        }

        for i, (title, column) in enumerate(METRICS_TO_PLOT.items()):
            ax = axes[i]
            
            # Plot Baseline first if it exists
            if 'Baseline' in loaded_data:
                df_base = loaded_data['Baseline']
                if column in df_base.columns:
                    ax.plot(df_base['epoch'], df_base[column], color='black', linestyle='--', label='Baseline', linewidth=2.5)

            # Plot other models
            for name, df_aug in loaded_data.items():
                if name == 'Baseline':
                    continue
                if column in df_aug.columns:
                    # Use the custom color map, with a default color of 'gray'
                    color = aug_color_map.get(name, 'gray')
                    ax.plot(df_aug['epoch'], df_aug[column], color=color, label=name, linewidth=2)

            ax.set_title(title, fontsize=14)
            ax.set_ylabel('Score')
            ax.grid(True, linestyle=':', alpha=0.7)
            ax.legend()

        axes[-1].set_xlabel('Epoch', fontsize=12)
        plt.tight_layout(rect=[0, 0.03, 1, 0.96])
        output_path = os.path.join(OUTPUT_PLOT_DIRECTORY, "Comparison_RandomWindowing.png")
        plt.savefig(output_path)
        plt.close(fig)

        print(f"\n✅ Plot successfully saved to: {output_path}")

    except Exception as e:
        print(f"\n❌ ERROR: An unexpected error occurred during plotting: {e}")

✅ Configuration loaded.

🔍 Starting automatic model discovery...
  - Baseline directory to exclude: '11-r-b-obb-w-yolov8n'
  - Found and added 'RandWin (Online)'
  - Found and added 'RandWin (Offline)'

✅ Discovery complete. Models to compare: ['Baseline', 'RandWin (Online)', 'RandWin (Offline)']
  - Successfully loaded data for 'Baseline'
  - Successfully loaded data for 'RandWin (Online)'
  - Successfully loaded data for 'RandWin (Offline)'

🖼️  Generating comparison plot...

✅ Plot successfully saved to: /kaggle/working/comparison_plots/Comparison_RandomWindowing.png
